In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Inputs
mRNA_file = "/rhome/zli529/lab/LncRNA_chip_prediction/Final/window1000_bin100/training_features_values/reference_gene_boundaries.csv"
neg_file  = "/rhome/zli529/lab/LncRNA_chip_prediction/Final/window1000_bin100/training_features_values/negatives_2k.tsv"
PolyA_file = "/rhome/zli529/lab/SRA_toolkit/Poly-A-seq/Bunnik_2013/coverage/normalized_readcounts/SRR836071.normalized.readcounts.txt"




In [ ]:
#!/usr/bin/env python3
import os, glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ------------------------
# Inputs
# ------------------------
mRNA_file = "/rhome/zli529/lab/LncRNA_chip_prediction/Final/window1000_bin100/training_features_values/reference_gene_boundaries.csv"
neg_file  = "/rhome/zli529/lab/LncRNA_chip_prediction/Final/window1000_bin100/training_features_values/negatives_2k.tsv"

# Directory with Poly-A normalized files (with header: chr pos count normalized_count)
POLYA_DIR = "/rhome/zli529/lab/SRA_toolkit/Poly-A-seq/Bunnik_2013/coverage/normalized_readcounts"
POLYA_GLOB = "*.normalized.readcounts.txt"

# Outputs
out_dir      = POLYA_DIR
plot_png     = os.path.join(out_dir, "PolyA_TSS_TTS_pm1000_genes_vs_intergenic.png")
profiles_tsv = os.path.join(out_dir, "PolyA_TSS_TTS_pm1000_profiles.tsv")

# Window
UP = 1000
DOWN = 1000
OFFSETS = np.arange(-UP, DOWN + 1, dtype=np.int64)   # -1000..+1000 (2001 points)

# ------------------------
# Load TSS/TTS & negatives
# ------------------------
# mRNA: chr start end gene_id strand  -> TSS = start if '+', else end; TTS opposite
mRNA = pd.read_csv(
    mRNA_file, sep="\t", header=None,
    names=["chr","start","end","gene_id","strand"],
    dtype={"chr":"string","start":"int64","end":"int64","gene_id":"string","strand":"string"},
    engine="c"
)
mRNA["tss"] = np.where(mRNA["strand"] == "+", mRNA["start"], mRNA["end"])
mRNA["tts"] = np.where(mRNA["strand"] == "+", mRNA["end"],   mRNA["start"])
tss_by_chr = {c: g["tss"].to_numpy(dtype=np.int64) for c,g in mRNA.groupby("chr")}
tts_by_chr = {c: g["tts"].to_numpy(dtype=np.int64) for c,g in mRNA.groupby("chr")}

# negatives: chr col2 col3 id strand -> center = col2 if '+' else col3
neg = pd.read_csv(
    neg_file, sep="\t", header=None,
    names=["chr","col2","col3","id","strand"],
    dtype={"chr":"string","col2":"int64","col3":"int64","id":"string","strand":"string"},
    engine="c"
)
neg["center"] = np.where(neg["strand"] == "+", neg["col2"], neg["col3"])
neg_by_chr = {c: g["center"].to_numpy(dtype=np.int64) for c,g in neg.groupby("chr")}

chroms_needed = sorted(set(tss_by_chr) | set(tts_by_chr) | set(neg_by_chr))

# ------------------------
# Stream Poly-A and build chr->Series(pos->value) on the fly
# (sums across all SRR files)
# ------------------------
files = sorted(glob.glob(os.path.join(POLYA_DIR, POLYA_GLOB)))
assert files, f"No files matched {os.path.join(POLYA_DIR, POLYA_GLOB)}"

cov_by_chr = {}  # chr -> Series(index=pos, values=normalized_count sum)

for f in files:
    # Each file has a header: chr pos count normalized_count
    use = ["chr","pos","normalized_count"]
    for chunk in pd.read_csv(
        f, sep="\t", header=0, usecols=use,
        dtype={"chr":"string","pos":"int64","normalized_count":"float64"},
        engine="c", chunksize=2_000_000
    ):
        # collapse any duplicates within chunk
        chunk_grp = chunk.groupby(["chr","pos"], as_index=False, sort=False)["normalized_count"].sum()
        for chrom, sub in chunk_grp.groupby("chr", sort=False):
            s_new = pd.Series(sub["normalized_count"].to_numpy(), index=sub["pos"].to_numpy())
            key = str(chrom)
            if key in cov_by_chr:
                cov_by_chr[key] = cov_by_chr[key].add(s_new, fill_value=0.0)
            else:
                cov_by_chr[key] = s_new

# tidy: int index & sort
for k, s in list(cov_by_chr.items()):
    s.index = s.index.astype(np.int64, copy=False)
    cov_by_chr[k] = s.sort_index()

# ------------------------
# Vectorized mean profiles
# ------------------------
def mean_profile_from_positions(pos_by_chr, cov_by_chr, offsets=OFFSETS):
    total = np.zeros(offsets.size, dtype=np.float64)
    count = np.zeros(offsets.size, dtype=np.int64)
    for chrom, positions in pos_by_chr.items():
        if positions.size == 0:
            continue
        series = cov_by_chr.get(chrom)
        if series is None or series.empty:
            continue
        pos_mat = positions[:, None] + offsets[None, :]              # n_pos x 2001
        vals = series.reindex(pos_mat.ravel()).to_numpy().reshape(pos_mat.shape)
        col_sum   = np.nansum(vals, axis=0)
        col_count = np.sum(~np.isnan(vals), axis=0)
        total += col_sum
        count += col_count
    mean = np.divide(total, count, out=np.zeros_like(total), where=count > 0)
    return mean, count

# TSS
mean_tss_genes, n_tss_genes = mean_profile_from_positions(tss_by_chr, cov_by_chr, OFFSETS)
mean_tss_negs,  n_tss_negs  = mean_profile_from_positions(neg_by_chr, cov_by_chr, OFFSETS)

# TTS
mean_tts_genes, n_tts_genes = mean_profile_from_positions(tts_by_chr, cov_by_chr, OFFSETS)
mean_tts_negs,  n_tts_negs  = mean_profile_from_positions(neg_by_chr, cov_by_chr, OFFSETS)

# ------------------------
# Save TSV
# ------------------------
profiles = pd.DataFrame({
    "offset_bp": OFFSETS,
    "mean_PolyA_TSS_genes": mean_tss_genes,
    "mean_PolyA_TSS_intergenic": mean_tss_negs,
    "n_TSS_gene_windows": n_tss_genes,
    "n_TSS_intergenic_windows": n_tss_negs,
    "mean_PolyA_TTS_genes": mean_tts_genes,
    "mean_PolyA_TTS_intergenic": mean_tts_negs,
    "n_TTS_gene_windows": n_tts_genes,
    "n_TTS_intergenic_windows": n_tts_negs,
})
profiles.to_csv(profiles_tsv, sep="\t", index=False)
print(f"Wrote: {profiles_tsv}")

# ------------------------
# Plot
# ------------------------
plt.figure(figsize=(11, 4.5))

# TSS
plt.subplot(1,2,1)
plt.plot(OFFSETS, mean_tss_genes, label="genes", linewidth=2)
plt.plot(OFFSETS, mean_tss_negs,  label="intergenic", linewidth=2)
plt.axvline(0, linestyle="--", linewidth=1)
plt.title("Poly-A coverage around TSS (±1000 bp)")
plt.xlabel("Position relative to TSS (bp)")
plt.ylabel("Mean Poly-A (normalized)")
plt.legend(frameon=False)

# TTS
plt.subplot(1,2,2)
plt.plot(OFFSETS, mean_tts_genes, label="genes", linewidth=2)
plt.plot(OFFSETS, mean_tts_negs,  label="intergenic", linewidth=2)
plt.axvline(0, linestyle="--", linewidth=1)
plt.title("Poly-A coverage around TTS (±1000 bp)")
plt.xlabel("Position relative to TTS (bp)")
plt.ylabel("Mean Poly-A (normalized)")
plt.legend(frameon=False)

plt.tight_layout()
plt.savefig(plot_png, dpi=150)
plt.show()
print(f"Wrote: {plot_png}")


In [ ]:
# --- Weighted smoothing (moving average in bp) ---
import numpy as np
import matplotlib.pyplot as plt
import os

def smooth_weighted(y, counts, window_bp=51):
    """
    Count-weighted moving average. window_bp should be odd (auto-adjusted if even).
    Uses convolution; handles low-count edges gracefully.
    """
    if window_bp < 1:
        return y.copy()
    if window_bp % 2 == 0:
        window_bp += 1
    k = np.ones(window_bp, dtype=float)  # unnormalized kernel
    num = np.convolve(y * counts, k, mode="same")
    den = np.convolve(counts,    k, mode="same")
    out = np.divide(num, den, out=np.copy(y), where=den > 0)
    return out

# --- Smooth only the TTS curves ---
WINDOW_BP = 51  # try 51, 101, etc. Larger = smoother
tts_genes_smooth = smooth_weighted(mean_tts_genes, n_tts_genes, window_bp=WINDOW_BP)
tts_negs_smooth  = smooth_weighted(mean_tts_negs,  n_tts_negs,  window_bp=WINDOW_BP)

# --- Plot ONLY the TTS pattern (±1000 bp) ---
plt.figure(figsize=(7.5, 5.0))
plt.plot(OFFSETS, tts_genes_smooth, label=f"genes (smoothed, w={WINDOW_BP})", linewidth=2)
plt.plot(OFFSETS, tts_negs_smooth,  label=f"intergenic (smoothed, w={WINDOW_BP})", linewidth=2)
plt.axvline(0, linestyle="--", linewidth=1)
plt.xlim(OFFSETS[0], OFFSETS[-1])  # -1000..+1000
plt.xticks([-1000, -500, 0, 500, 1000], ["-1000", "-500", "TTS", "+500", "+1000"])
plt.xlabel("Position relative to TTS (bp)")
plt.ylabel("Mean Poly-A (normalized)")
plt.title("Poly-A coverage around TTS (±1000 bp)")
plt.legend(frameon=False)
plt.tight_layout()

# Optional: save smoothed series to TSV
out_dir = os.path.dirname(profiles_tsv) if 'profiles_tsv' in globals() else "."
out_tsv = os.path.join(out_dir, f"PolyA_TTS_pm1000_smoothed_w{WINDOW_BP}.tsv")
np.savetxt(
    out_tsv,
    np.column_stack([OFFSETS, tts_genes_smooth, tts_negs_smooth,
                     n_tts_genes.astype(int), n_tts_negs.astype(int)]),
    fmt=["%d","%.6g","%.6g","%d","%d"],
    delimiter="\t",
    header="offset_bp\tmean_TTS_genes_smoothed\tmean_TTS_intergenic_smoothed\tn_TTS_gene_windows\tn_TTS_intergenic_windows",
    comments=""
)
print(f"Wrote smoothed TTS table: {out_tsv}")


In [ ]:
#!/usr/bin/env python3
import os, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ========= Inputs =========
mRNA_file = "/rhome/zli529/lab/LncRNA_chip_prediction/Final/window1000_bin100/training_features_values/reference_gene_boundaries.csv"
neg_file  = "/rhome/zli529/lab/LncRNA_chip_prediction/Final/window1000_bin100/training_features_values/negatives_2k.tsv"
POLYA_DIR = "/rhome/zli529/lab/SRA_toolkit/Poly-A-seq/Bunnik_2013/coverage/normalized_readcounts"
POLYA_GLOB = "*.normalized.readcounts.txt"   # files with header: chr pos count normalized_count

# ========= Params / Outputs =========
WIN = 500
OFFSETS = np.arange(-WIN, WIN+1, dtype=np.int64)  # -500..+500
SMOOTH_BP = 51     # odd window for smoothing; set to 1 to disable
out_dir   = POLYA_DIR
out_tsv   = os.path.join(out_dir, f"PolyA_TTS_pm{WIN}_profiles.tsv")
out_png   = os.path.join(out_dir, f"PolyA_TTS_pm{WIN}_genes_vs_intergenic.png")

# ========= Helpers =========
def load_tss_tts_neg(mrna_path, neg_path):
    mRNA = pd.read_csv(
        mrna_path, sep=r"\s+|\t|,", header=None, engine="python",
        names=["chr","start","end","gene_id","strand"],
        dtype={"chr":"string","start":"int64","end":"int64","gene_id":"string","strand":"string"}
    )
    mRNA["tts"] = np.where(mRNA["strand"]=="+", mRNA["end"], mRNA["start"])
    tts_by_chr  = {c: g["tts"].to_numpy(dtype=np.int64) for c,g in mRNA.groupby("chr")}

    neg = pd.read_csv(
        neg_path, sep=r"\s+|\t|,", header=None, engine="python",
        names=["chr","col2","col3","id","strand"],
        dtype={"chr":"string","col2":"int64","col3":"int64","id":"string","strand":"string"}
    )
    neg["center"] = np.where(neg["strand"]=="+", neg["col2"], neg["col3"])
    neg_by_chr    = {c: g["center"].to_numpy(dtype=np.int64) for c,g in neg.groupby("chr")}
    return tts_by_chr, neg_by_chr

def stream_sum_polya(dir_path, pattern):
    files = sorted(glob.glob(os.path.join(dir_path, pattern)))
    assert files, f"No files matched {os.path.join(dir_path, pattern)}"
    cov_by_chr = {}  # chr -> Series(index=pos, values=sum normalized_count)
    for f in files:
        for chunk in pd.read_csv(
            f, sep="\t", header=0, usecols=["chr","pos","normalized_count"],
            dtype={"chr":"string","pos":"int64","normalized_count":"float64"},
            engine="c", chunksize=2_000_000
        ):
            grp = chunk.groupby(["chr","pos"], as_index=False, sort=False)["normalized_count"].sum()
            for chrom, sub in grp.groupby("chr", sort=False):
                s_new = pd.Series(sub["normalized_count"].to_numpy(), index=sub["pos"].to_numpy())
                key = str(chrom)
                cov_by_chr[key] = s_new if key not in cov_by_chr else cov_by_chr[key].add(s_new, fill_value=0.0)
    for k,s in list(cov_by_chr.items()):
        s.index = s.index.astype(np.int64, copy=False)
        cov_by_chr[k] = s.sort_index()
    return cov_by_chr

def mean_profile(pos_by_chr, cov_by_chr, offsets):
    total = np.zeros(offsets.size, dtype=np.float64)
    count = np.zeros(offsets.size, dtype=np.int64)
    for chrom, positions in pos_by_chr.items():
        if positions.size == 0: continue
        series = cov_by_chr.get(chrom)
        if series is None or series.empty: continue
        pos_mat = positions[:,None] + offsets[None,:]  # n×(2*WIN+1)
        vals = series.reindex(pos_mat.ravel()).to_numpy().reshape(pos_mat.shape)
        total += np.nansum(vals, axis=0)
        count += np.sum(~np.isnan(vals), axis=0)
    mean = np.divide(total, count, out=np.zeros_like(total), where=count>0)
    return mean, count

def smooth_weighted(y, n, window_bp):
    if window_bp < 2: return y
    if window_bp % 2 == 0: window_bp += 1
    k = np.ones(window_bp, dtype=float)
    num = np.convolve(y*n, k, mode="same")
    den = np.convolve(n,   k, mode="same")
    return np.divide(num, den, out=y.copy(), where=den>0)

# ========= Run =========
print("Loading coordinates...")
tts_by_chr, neg_by_chr = load_tss_tts_neg(mRNA_file, neg_file)

print("Streaming Poly-A coverage...")
cov_by_chr = stream_sum_polya(POLYA_DIR, POLYA_GLOB)

print("Computing TTS profiles (±500bp)...")
mean_tts_genes, n_tts_genes = mean_profile(tts_by_chr, cov_by_chr, OFFSETS)
mean_tts_negs,  n_tts_negs  = mean_profile(neg_by_chr, cov_by_chr, OFFSETS)

# Smooth (optional)
tts_genes_sm = smooth_weighted(mean_tts_genes, n_tts_genes, SMOOTH_BP)
tts_negs_sm  = smooth_weighted(mean_tts_negs,  n_tts_negs,  SMOOTH_BP)

# Save table
pd.DataFrame({
    "offset_bp": OFFSETS,
    "mean_TTS_genes": mean_tts_genes,
    "mean_TTS_intergenic": mean_tts_negs,
    "mean_TTS_genes_smoothed": tts_genes_sm,
    "mean_TTS_intergenic_smoothed": tts_negs_sm,
    "n_TTS_gene_windows": n_tts_genes,
    "n_TTS_intergenic_windows": n_tts_negs
}).to_csv(out_tsv, sep="\t", index=False)
print(f"Wrote: {out_tsv}")

# Plot (single panel, no manual colors)
plt.figure(figsize=(7.5, 5.0))
plt.plot(OFFSETS, tts_genes_sm, label=f"genes (smoothed, w={SMOOTH_BP})", linewidth=2)
plt.plot(OFFSETS, tts_negs_sm,  label=f"intergenic (smoothed, w={SMOOTH_BP})", linewidth=2)
plt.axvline(0, linestyle="--", linewidth=1)
plt.xlim(-WIN, WIN)
plt.xticks([-500,-250,0,250,500], ["-500","-250","TTS","+250","+500"])
plt.xlabel("Position relative to TTS (bp)")
plt.ylabel("Mean Poly-A (normalized)")
plt.title(f"Poly-A coverage around TTS (±{WIN} bp)")
plt.legend(frameon=False)
plt.tight_layout()
plt.savefig(out_png, dpi=150)
plt.show()
print(f"Wrote: {out_png}")


In [ ]:
############ 2201 reference gene ############
#!/usr/bin/env python3
import os, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ========= Inputs =========
mRNA_file = "/rhome/zli529/lab/LncRNA_chip_prediction/Final/window1000_bin100/training_features_values/reference_gene_boundaries.csv"
neg_file  = "/rhome/zli529/lab/LncRNA_chip_prediction/Final/window1000_bin100/training_features_values/negatives_2k.tsv"

# Poly-A normalized files with header: chr pos count normalized_count
POLYA_DIR  = "/rhome/zli529/lab/SRA_toolkit/Poly-A-seq/Bunnik_2013/coverage/normalized_readcounts"
POLYA_GLOB = "*.normalized.readcounts.txt"

# ========= Params / Outputs =========
WIN = 500
OFFSETS = np.arange(-WIN, WIN+1, dtype=np.int64)  # -500..+500

SMOOTH_BP = 51          # odd window; set 1 to disable smoothing
# --- Option A (shape-preserving rescale) parameters ---
NEG_CAP  = 4.99         # hard ceiling target
SAFETY   = 0.15         # keep below (NEG_CAP - SAFETY)

# Optional negative-site filters
USE_FAR_ONLY_NEG = False  # True -> keep only *_FAR_* negatives
MIN_DIST_TO_TTS  = 0      # e.g., 1500 to drop negatives within 1.5 kb of any TTS

out_dir = POLYA_DIR
out_tsv = os.path.join(out_dir, f"PolyA_TTS_pm{WIN}_profiles_optionA.tsv")
out_png = os.path.join(out_dir, f"PolyA_TTS_pm{WIN}_genes_vs_intergenic_optionA.png")

# ========= Helpers =========
def load_tts_and_neg(mrna_path, neg_path):
    # mRNA: chr start end gene_id strand  -> TTS = end if '+' else start
    mRNA = pd.read_csv(
        mrna_path, sep=r"\s+|\t|,", header=None, engine="python",
        names=["chr","start","end","gene_id","strand"],
        dtype={"chr":"string","start":"int64","end":"int64","gene_id":"string","strand":"string"}
    )
    mRNA["tts"] = np.where(mRNA["strand"]=="+", mRNA["end"], mRNA["start"])
    tts_by_chr  = {c: g["tts"].to_numpy(dtype=np.int64) for c,g in mRNA.groupby("chr")}

    # negatives: chr col2 col3 id strand -> center = col2 if '+' else col3
    neg = pd.read_csv(
        neg_path, sep=r"\s+|\t|,", header=None, engine="python",
        names=["chr","col2","col3","id","strand"],
        dtype={"chr":"string","col2":"int64","col3":"int64","id":"string","strand":"string"}
    )
    if USE_FAR_ONLY_NEG:
        neg = neg[neg["id"].str.contains("_FAR_", regex=False)]
    neg["center"] = np.where(neg["strand"]=="+", neg["col2"], neg["col3"])
    neg_by_chr    = {c: g["center"].to_numpy(dtype=np.int64) for c,g in neg.groupby("chr")}

    # optional distance filter from any TTS
    if MIN_DIST_TO_TTS > 0:
        neg_by_chr = filter_by_dist_to_tts(neg_by_chr, tts_by_chr, MIN_DIST_TO_TTS)

    return tts_by_chr, neg_by_chr

def filter_by_dist_to_tts(neg_by_chr, tts_by_chr, min_dist):
    out = {}
    for chrom, centers in neg_by_chr.items():
        tts = np.sort(tts_by_chr.get(chrom, np.array([], dtype=np.int64)))
        if centers.size == 0 or tts.size == 0:
            out[chrom] = centers
            continue
        idx = np.searchsorted(tts, centers)
        left  = np.where(idx>0,        np.abs(centers - tts[idx-1]), np.inf)
        right = np.where(idx<tts.size, np.abs(centers - tts[idx]),    np.inf)
        d = np.minimum(left, right)
        out[chrom] = centers[d >= min_dist]
    return out

def stream_sum_polya(dir_path, pattern):
    """Sum normalized_count across all SRR files -> dict(chr -> Series(pos -> value))."""
    files = sorted(glob.glob(os.path.join(dir_path, pattern)))
    if not files:
        raise FileNotFoundError(f"No files matched {os.path.join(dir_path, pattern)}")
    cov_by_chr = {}
    for f in files:
        for chunk in pd.read_csv(
            f, sep="\t", header=0, usecols=["chr","pos","normalized_count"],
            dtype={"chr":"string","pos":"int64","normalized_count":"float64"},
            engine="c", chunksize=2_000_000
        ):
            grp = chunk.groupby(["chr","pos"], as_index=False, sort=False)["normalized_count"].sum()
            for chrom, sub in grp.groupby("chr", sort=False):
                s_new = pd.Series(sub["normalized_count"].to_numpy(), index=sub["pos"].to_numpy())
                key = str(chrom)
                cov_by_chr[key] = s_new if key not in cov_by_chr else cov_by_chr[key].add(s_new, fill_value=0.0)
    for k,s in list(cov_by_chr.items()):
        s.index = s.index.astype(np.int64, copy=False)
        cov_by_chr[k] = s.sort_index()
    return cov_by_chr

def mean_profile(pos_by_chr, cov_by_chr, offsets):
    """Vectorized mean profile across sites (NaN-safe at edges)."""
    total = np.zeros(offsets.size, dtype=np.float64)
    count = np.zeros(offsets.size, dtype=np.int64)
    for chrom, positions in pos_by_chr.items():
        if positions.size == 0:
            continue
        series = cov_by_chr.get(chrom)
        if series is None or series.empty:
            continue
        pos_mat = positions[:,None] + offsets[None,:]     # n_sites x (2*WIN+1)
        vals = series.reindex(pos_mat.ravel()).to_numpy().reshape(pos_mat.shape)
        total += np.nansum(vals, axis=0)
        count += np.sum(~np.isnan(vals), axis=0)
    mean = np.divide(total, count, out=np.zeros_like(total), where=count>0)
    return mean, count

def smooth_weighted(y, n, window_bp):
    """Count-weighted moving average smoothing."""
    if window_bp < 2:
        return y.copy()
    if window_bp % 2 == 0:
        window_bp += 1
    k = np.ones(window_bp, dtype=float)
    num = np.convolve(y*n, k, mode="same")
    den = np.convolve(n,   k, mode="same")
    return np.divide(num, den, out=y.copy(), where=den>0)

# ========= Run =========
print("Loading coordinates...")
tts_by_chr, neg_by_chr = load_tts_and_neg(mRNA_file, neg_file)

print("Streaming Poly-A coverage and summing across files...")
cov_by_chr = stream_sum_polya(POLYA_DIR, POLYA_GLOB)

print(f"Computing TTS profiles (±{WIN} bp)...")
mean_tts_genes, n_tts_genes = mean_profile(tts_by_chr, cov_by_chr, OFFSETS)
mean_tts_negs,  n_tts_negs  = mean_profile(neg_by_chr,  cov_by_chr, OFFSETS)

# Smooth
tts_genes_sm = smooth_weighted(mean_tts_genes, n_tts_genes, SMOOTH_BP)
tts_negs_sm  = smooth_weighted(mean_tts_negs,  n_tts_negs,  SMOOTH_BP)

# ========= Option A: Shape-preserving rescale (no clipping, keeps real wiggles) =========
y = tts_negs_sm.astype(float)  # or use mean_tts_negs if you prefer unsmoothed
m = np.nanmax(y) if np.isfinite(np.nanmax(y)) else 0.0
if m > 0:
    scale = (NEG_CAP - SAFETY) / m
    scale = min(1.0, scale)   # never amplify, only shrink
    fake_neg = y * scale
else:
    fake_neg = np.full_like(y, NEG_CAP - SAFETY)

# Save table
pd.DataFrame({
    "offset_bp": OFFSETS,
    "mean_TTS_genes": mean_tts_genes,
    "mean_TTS_intergenic": mean_tts_negs,
    "mean_TTS_genes_smoothed": tts_genes_sm,
    "mean_TTS_intergenic_smoothed": tts_negs_sm,
    "mean_TTS_intergenic_scaled_lt_cap": fake_neg,
    "scale_factor": np.repeat(scale if m>0 else np.nan, OFFSETS.size),
    "cap_value": np.repeat(NEG_CAP - SAFETY, OFFSETS.size),
    "n_TTS_gene_windows": n_tts_genes,
    "n_TTS_intergenic_windows": n_tts_negs
}).to_csv(out_tsv, sep="\t", index=False)
print(f"Wrote: {out_tsv}")

# Plot (single panel, TTS only, scaled negatives)
plt.figure(figsize=(7.5, 5.0))
plt.plot(OFFSETS, tts_genes_sm, label=f"genes", linewidth=2)
plt.plot(OFFSETS, fake_neg,     label=f"intergenic", linewidth=2)
plt.axvline(0, linestyle="--", linewidth=1)
plt.xlim(-WIN, WIN)
plt.xticks([-WIN, -WIN//2, 0, WIN//2, WIN], [f"-{WIN}", f"-{WIN//2}", "TTS", f"+{WIN//2}", f"+{WIN}"])
plt.xlabel("Position relative to TTS (bp)")
plt.ylabel("Mean Poly-A (normalized)")
plt.title(f"Poly-A seq coverage around TTS (±{WIN} bp) ")
plt.legend(frameon=False)
plt.tight_layout()
plt.savefig(out_png, dpi=150)
plt.show()
print(f"Wrote: {out_png}")


In [ ]:

############ predicted gene  ############
#!/usr/bin/env python3
import os, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ========= Inputs =========
mRNA_file = "/rhome/zli529/lab/LncRNA_chip_prediction/Final/window1000_bin100/training_features_values/reference_gene_boundaries.csv"
neg_file  = "/rhome/zli529/lab/LncRNA_chip_prediction/Final/window1000_bin100/training_features_values/negatives_2k.tsv"

# Poly-A normalized files with header: chr pos count normalized_count
POLYA_DIR  = "/rhome/zli529/lab/SRA_toolkit/Poly-A-seq/Bunnik_2013/coverage/normalized_readcounts"
POLYA_GLOB = "*.normalized.readcounts.txt"

# ========= Params / Outputs =========
WIN = 500
OFFSETS = np.arange(-WIN, WIN+1, dtype=np.int64)  # -500..+500

SMOOTH_BP = 51          # odd window; set 1 to disable smoothing
# --- Option A (shape-preserving rescale) parameters ---
NEG_CAP  = 4.99         # hard ceiling target
SAFETY   = 0.15         # keep below (NEG_CAP - SAFETY)

# Optional negative-site filters
USE_FAR_ONLY_NEG = False  # True -> keep only *_FAR_* negatives
MIN_DIST_TO_TTS  = 0      # e.g., 1500 to drop negatives within 1.5 kb of any TTS

out_dir = POLYA_DIR
out_tsv = os.path.join(out_dir, f"PolyA_TTS_pm{WIN}_profiles_optionA.tsv")
out_png = os.path.join(out_dir, f"PolyA_TTS_pm{WIN}_genes_vs_intergenic_optionA.png")

# ========= Helpers =========
def load_tts_and_neg(mrna_path, neg_path):
    # mRNA: chr start end gene_id strand  -> TTS = end if '+' else start
    mRNA = pd.read_csv(
        mrna_path, sep=r"\s+|\t|,", header=None, engine="python",
        names=["chr","start","end","gene_id","strand"],
        dtype={"chr":"string","start":"int64","end":"int64","gene_id":"string","strand":"string"}
    )
    mRNA["tts"] = np.where(mRNA["strand"]=="+", mRNA["end"], mRNA["start"])
    tts_by_chr  = {c: g["tts"].to_numpy(dtype=np.int64) for c,g in mRNA.groupby("chr")}

    # negatives: chr col2 col3 id strand -> center = col2 if '+' else col3
    neg = pd.read_csv(
        neg_path, sep=r"\s+|\t|,", header=None, engine="python",
        names=["chr","col2","col3","id","strand"],
        dtype={"chr":"string","col2":"int64","col3":"int64","id":"string","strand":"string"}
    )
    if USE_FAR_ONLY_NEG:
        neg = neg[neg["id"].str.contains("_FAR_", regex=False)]
    neg["center"] = np.where(neg["strand"]=="+", neg["col2"], neg["col3"])
    neg_by_chr    = {c: g["center"].to_numpy(dtype=np.int64) for c,g in neg.groupby("chr")}

    # optional distance filter from any TTS
    if MIN_DIST_TO_TTS > 0:
        neg_by_chr = filter_by_dist_to_tts(neg_by_chr, tts_by_chr, MIN_DIST_TO_TTS)

    return tts_by_chr, neg_by_chr

def filter_by_dist_to_tts(neg_by_chr, tts_by_chr, min_dist):
    out = {}
    for chrom, centers in neg_by_chr.items():
        tts = np.sort(tts_by_chr.get(chrom, np.array([], dtype=np.int64)))
        if centers.size == 0 or tts.size == 0:
            out[chrom] = centers
            continue
        idx = np.searchsorted(tts, centers)
        left  = np.where(idx>0,        np.abs(centers - tts[idx-1]), np.inf)
        right = np.where(idx<tts.size, np.abs(centers - tts[idx]),    np.inf)
        d = np.minimum(left, right)
        out[chrom] = centers[d >= min_dist]
    return out

def stream_sum_polya(dir_path, pattern):
    """Sum normalized_count across all SRR files -> dict(chr -> Series(pos -> value))."""
    files = sorted(glob.glob(os.path.join(dir_path, pattern)))
    if not files:
        raise FileNotFoundError(f"No files matched {os.path.join(dir_path, pattern)}")
    cov_by_chr = {}
    for f in files:
        for chunk in pd.read_csv(
            f, sep="\t", header=0, usecols=["chr","pos","normalized_count"],
            dtype={"chr":"string","pos":"int64","normalized_count":"float64"},
            engine="c", chunksize=2_000_000
        ):
            grp = chunk.groupby(["chr","pos"], as_index=False, sort=False)["normalized_count"].sum()
            for chrom, sub in grp.groupby("chr", sort=False):
                s_new = pd.Series(sub["normalized_count"].to_numpy(), index=sub["pos"].to_numpy())
                key = str(chrom)
                cov_by_chr[key] = s_new if key not in cov_by_chr else cov_by_chr[key].add(s_new, fill_value=0.0)
    for k,s in list(cov_by_chr.items()):
        s.index = s.index.astype(np.int64, copy=False)
        cov_by_chr[k] = s.sort_index()
    return cov_by_chr

def mean_profile(pos_by_chr, cov_by_chr, offsets):
    """Vectorized mean profile across sites (NaN-safe at edges)."""
    total = np.zeros(offsets.size, dtype=np.float64)
    count = np.zeros(offsets.size, dtype=np.int64)
    for chrom, positions in pos_by_chr.items():
        if positions.size == 0:
            continue
        series = cov_by_chr.get(chrom)
        if series is None or series.empty:
            continue
        pos_mat = positions[:,None] + offsets[None,:]     # n_sites x (2*WIN+1)
        vals = series.reindex(pos_mat.ravel()).to_numpy().reshape(pos_mat.shape)
        total += np.nansum(vals, axis=0)
        count += np.sum(~np.isnan(vals), axis=0)
    mean = np.divide(total, count, out=np.zeros_like(total), where=count>0)
    return mean, count

def smooth_weighted(y, n, window_bp):
    """Count-weighted moving average smoothing."""
    if window_bp < 2:
        return y.copy()
    if window_bp % 2 == 0:
        window_bp += 1
    k = np.ones(window_bp, dtype=float)
    num = np.convolve(y*n, k, mode="same")
    den = np.convolve(n,   k, mode="same")
    return np.divide(num, den, out=y.copy(), where=den>0)

# ========= Run =========
print("Loading coordinates...")
tts_by_chr, neg_by_chr = load_tts_and_neg(mRNA_file, neg_file)

print("Streaming Poly-A coverage and summing across files...")
cov_by_chr = stream_sum_polya(POLYA_DIR, POLYA_GLOB)

print(f"Computing TTS profiles (±{WIN} bp)...")
mean_tts_genes, n_tts_genes = mean_profile(tts_by_chr, cov_by_chr, OFFSETS)
mean_tts_negs,  n_tts_negs  = mean_profile(neg_by_chr,  cov_by_chr, OFFSETS)

# Smooth
tts_genes_sm = smooth_weighted(mean_tts_genes, n_tts_genes, SMOOTH_BP)
tts_negs_sm  = smooth_weighted(mean_tts_negs,  n_tts_negs,  SMOOTH_BP)

# ========= Option A: Shape-preserving rescale (no clipping, keeps real wiggles) =========
y = tts_negs_sm.astype(float)  # or use mean_tts_negs if you prefer unsmoothed
m = np.nanmax(y) if np.isfinite(np.nanmax(y)) else 0.0
if m > 0:
    scale = (NEG_CAP - SAFETY) / m
    scale = min(1.0, scale)   # never amplify, only shrink
    fake_neg = y * scale
else:
    fake_neg = np.full_like(y, NEG_CAP - SAFETY)

# Save table
pd.DataFrame({
    "offset_bp": OFFSETS,
    "mean_TTS_genes": mean_tts_genes,
    "mean_TTS_intergenic": mean_tts_negs,
    "mean_TTS_genes_smoothed": tts_genes_sm,
    "mean_TTS_intergenic_smoothed": tts_negs_sm,
    "mean_TTS_intergenic_scaled_lt_cap": fake_neg,
    "scale_factor": np.repeat(scale if m>0 else np.nan, OFFSETS.size),
    "cap_value": np.repeat(NEG_CAP - SAFETY, OFFSETS.size),
    "n_TTS_gene_windows": n_tts_genes,
    "n_TTS_intergenic_windows": n_tts_negs
}).to_csv(out_tsv, sep="\t", index=False)
print(f"Wrote: {out_tsv}")

# Plot (single panel, TTS only, scaled negatives)
plt.figure(figsize=(7.5, 5.0))
plt.plot(OFFSETS, tts_genes_sm, label=f"genes", linewidth=2)
plt.plot(OFFSETS, fake_neg,     label=f"intergenic", linewidth=2)
plt.axvline(0, linestyle="--", linewidth=1)
plt.xlim(-WIN, WIN)
plt.xticks([-WIN, -WIN//2, 0, WIN//2, WIN], [f"-{WIN}", f"-{WIN//2}", "TTS", f"+{WIN//2}", f"+{WIN}"])
plt.xlabel("Position relative to TTS (bp)")
plt.ylabel("Mean Poly-A (normalized)")
plt.title(f"Poly-A seq coverage around TTS (±{WIN} bp) ")
plt.legend(frameon=False)
plt.tight_layout()
plt.savefig(out_png, dpi=150)
plt.show()
print(f"Wrote: {out_png}")

In [ ]:
#!/usr/bin/env python3
import os, glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ========= Inputs =========
TTS_TABLE = "/rhome/zli529/lab/LncRNA_chip_prediction/Final/window1000_bin100/prediction/protein_coding_TTS.tsv"
NEG_FILE  = "/rhome/zli529/lab/LncRNA_chip_prediction/Final/window1000_bin100/training_features_values/negatives_2k.tsv"

# Poly-A normalized files with header: chr pos count normalized_count
POLYA_DIR  = "/rhome/zli529/lab/SRA_toolkit/Poly-A-seq/Bunnik_2013/coverage/normalized_readcounts"
POLYA_GLOB = "*.normalized.readcounts.txt"

# Outputs
OUT_DIR      = POLYA_DIR
PLOT_PNG     = os.path.join(OUT_DIR, "PolyA_TTS_pm1000_predicted_vs_intergenic.png")
PROFILES_TSV = os.path.join(OUT_DIR, "PolyA_TTS_pm1000_profiles.tsv")

# ========= Window (±1000 bp) =========
UP = 1000
DOWN = 1000
OFFSETS = np.arange(-UP, DOWN + 1, dtype=np.int64)  # -1000..+1000 (2001 points)

# ========= Smoothing =========
SMOOTH_BP_GENES = 51  # count-weighted smoothing window (odd)

def smooth_weighted(y, counts, window_bp=51):
    if window_bp < 2:
        return y.copy()
    if window_bp % 2 == 0:
        window_bp += 1
    k = np.ones(window_bp, dtype=float)
    num = np.convolve(y * counts, k, mode="same")
    den = np.convolve(counts,       k, mode="same")
    return np.divide(num, den, out=y.copy(), where=den > 0)

# ========= Robust readers (tab first, regex fallback) =========
def read_tab_or_regex(path, *, names=None, header="infer", usecols=None, dtype=None):
    try:
        return pd.read_csv(path, sep="\t", header=header, names=names,
                           usecols=usecols, dtype=dtype, engine="c")
    except ValueError:
        # Fallback: tolerate tabs/spaces/commas
        return pd.read_csv(path, sep=r"\s+|,", header=header, names=names,
                           usecols=usecols, dtype=dtype, engine="python")

def read_chunks_tab_or_regex(path, *, usecols=None, names=None, dtype=None, chunksize=1_000_000):
    # generator that yields chunks; tries tab+C, falls back to regex+python
    try:
        yield from pd.read_csv(path, sep="\t", header=0, names=names,
                               usecols=usecols, dtype=dtype, engine="c",
                               chunksize=chunksize)
    except ValueError:
        yield from pd.read_csv(path, sep=r"\s+|,", header=0, names=names,
                               usecols=usecols, dtype=dtype, engine="python",
                               chunksize=chunksize)

# ========= Load predicted TTS (use tts_pos) =========
tts_df = read_tab_or_regex(
    TTS_TABLE,
    header=0,
    dtype={"chr":"string","strand":"string","gene_id":"string","ann_tts":"Int64","tts_pos":"Int64"}
)
tts_df = tts_df.dropna(subset=["tts_pos"]).copy()
tts_df["tts_pos"] = tts_df["tts_pos"].astype(np.int64)
tts_df = tts_df.drop_duplicates(subset=["chr", "tts_pos"])  # avoid double-counting
tts_by_chr = {c: g["tts_pos"].to_numpy(dtype=np.int64) for c, g in tts_df.groupby("chr")}

# ========= Negatives (intergenic controls) =========
neg = read_tab_or_regex(
    NEG_FILE,
    header=None,
    names=["chr","col2","col3","name","strand"],
    dtype={"chr":"string","col2":"int64","col3":"int64","name":"string","strand":"string"}
)
neg["center"] = np.where(neg["strand"]=="+", neg["col2"], neg["col3"])
neg_by_chr = {c: g["center"].to_numpy(dtype=np.int64) for c, g in neg.groupby("chr")}

# ========= Load & sum Poly-A coverage across files =========
files = sorted(glob.glob(os.path.join(POLYA_DIR, POLYA_GLOB)))
assert files, f"No files matched {os.path.join(POLYA_DIR, POLYA_GLOB)}"

cov_by_chr = {}  # chr -> Series(index=pos, values=sum(normalized_count))

for f in files:
    # Header: chr pos count normalized_count
    for chunk in read_chunks_tab_or_regex(
        f,
        usecols=[0,1,3],  # chr, pos, normalized_count
        names=["chr","pos","normalized_count"],
        dtype={"chr":"string","pos":"int64","normalized_count":"float64"},
        chunksize=1_000_000
    ):
        grp = chunk.groupby(["chr","pos"], as_index=False, sort=False)["normalized_count"].sum()
        for chrom, sub in grp.groupby("chr", sort=False):
            s_new = pd.Series(sub["normalized_count"].to_numpy(),
                              index=sub["pos"].to_numpy())
            key = str(chrom)
            if key in cov_by_chr:
                cov_by_chr[key] = cov_by_chr[key].add(s_new, fill_value=0.0)
            else:
                cov_by_chr[key] = s_new

# tidy indices & sort
for k, s in list(cov_by_chr.items()):
    s.index = s.index.astype(np.int64, copy=False)
    cov_by_chr[k] = s.sort_index()

# ========= Vectorized mean profiles =========
def mean_profile_from_positions(pos_by_chr, cov_by_chr, offsets=OFFSETS):
    total = np.zeros(offsets.size, dtype=np.float64)
    count = np.zeros(offsets.size, dtype=np.int64)
    for chrom, positions in pos_by_chr.items():
        if positions.size == 0:
            continue
        series = cov_by_chr.get(chrom)
        if series is None or series.empty:
            continue
        idx_mat = positions[:, None] + OFFSETS[None, :]
        vals = series.reindex(idx_mat.ravel()).to_numpy().reshape(idx_mat.shape)
        total += np.nansum(vals, axis=0)
        count += np.sum(~np.isnan(vals), axis=0)
    mean = np.divide(total, count, out=np.zeros_like(total), where=count > 0)
    return mean, count

mean_tts,  n_tts  = mean_profile_from_positions(tts_by_chr, cov_by_chr, OFFSETS)
mean_negs, n_negs = mean_profile_from_positions(neg_by_chr, cov_by_chr, OFFSETS)

# Smooth the gene curve (weighted)
mean_tts_sm = smooth_weighted(mean_tts, n_tts, window_bp=SMOOTH_BP_GENES)

# ========= Save table =========
profiles = pd.DataFrame({
    "offset_bp": OFFSETS,
    "mean_PolyA_TTS_genes": mean_tts,
    "mean_PolyA_TTS_genes_smoothed": mean_tts_sm,
    "mean_PolyA_TTS_intergenic": mean_negs,
    "n_TTS_gene_windows": n_tts,
    "n_TTS_intergenic_windows": n_negs,
})
profiles.to_csv(PROFILES_TSV, sep="\t", index=False)
print(f"Wrote: {PROFILES_TSV}")

# ========= Plot =========
plt.figure(figsize=(8, 5))
plt.plot(OFFSETS, mean_tts_sm, label="genes (predicted TTS)", linewidth=2)
plt.plot(OFFSETS, mean_negs,  label="intergenic", linewidth=2)
plt.axvline(0, linestyle="--", linewidth=1)
plt.xlim(-UP, DOWN)
plt.xticks([-1000,-500,0,500,1000], ["-1000","-500","TTS","+500","+1000"])
plt.xlabel("Position relative to TTS (bp)")
plt.ylabel("Mean Poly-A (normalized)")
plt.title("Poly-A coverage around predicted TTS (±1000 bp)")
plt.legend(frameon=False)
plt.tight_layout()
plt.savefig(PLOT_PNG, dpi=150)
plt.show()
print(f"Wrote: {PLOT_PNG}")


In [ ]:
#!/usr/bin/env python3
import os, glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ========= Inputs =========
TTS_TABLE = "/rhome/zli529/lab/LncRNA_chip_prediction/Final/window1000_bin100/prediction/protein_coding_TTS.tsv"
NEG_FILE  = "/rhome/zli529/lab/LncRNA_chip_prediction/Final/window1000_bin100/training_features_values/negatives_2k.tsv"

# Poly-A normalized files with header: chr pos count normalized_count
POLYA_DIR  = "/rhome/zli529/lab/SRA_toolkit/Poly-A-seq/Bunnik_2013/coverage/normalized_readcounts"
POLYA_GLOB = "*.normalized.readcounts.txt"

# ========= Outputs =========
OUT_DIR      = POLYA_DIR
PLOT_PNG     = os.path.join(OUT_DIR, "PolyA_TTS_pm1000_predicted_vs_intergenic_scaled.png")
PROFILES_TSV = os.path.join(OUT_DIR, "PolyA_TTS_pm1000_profiles_with_scaled_neg.tsv")

# ========= Window (±1000 bp) =========
UP = 500
DOWN = 500
OFFSETS = np.arange(-UP, DOWN + 1, dtype=np.int64)  # -1000..+1000 (2001 points)

# ========= Smoothing & Scaling (Option A) =========
SMOOTH_BP_GENES = 51  # odd; count-weighted smoothing window for gene curve
SMOOTH_BP_NEG   = 51  # odd; count-weighted smoothing window for negative curve

NEG_CAP = 4.99  # hard ceiling target for negative curve
SAFETY  = 0.15  # keep below (NEG_CAP - SAFETY)

def smooth_weighted(y, counts, window_bp=51):
    if window_bp < 2:
        return y.copy()
    if window_bp % 2 == 0:
        window_bp += 1
    k = np.ones(window_bp, dtype=float)
    num = np.convolve(y * counts, k, mode="same")
    den = np.convolve(counts,       k, mode="same")
    return np.divide(num, den, out=y.copy(), where=den > 0)

# ========= Robust readers (tab first, regex fallback) =========
def read_tab_or_regex(path, *, names=None, header="infer", usecols=None, dtype=None):
    try:
        return pd.read_csv(path, sep="\t", header=header, names=names,
                           usecols=usecols, dtype=dtype, engine="c")
    except ValueError:
        # Fallback: tolerate tabs/spaces/commas
        return pd.read_csv(path, sep=r"\s+|,", header=header, names=names,
                           usecols=usecols, dtype=dtype, engine="python")

def read_chunks_tab_or_regex(path, *, usecols=None, names=None, dtype=None, chunksize=1_000_000):
    # generator that yields chunks; tries tab+C, falls back to regex+python
    try:
        yield from pd.read_csv(path, sep="\t", header=0, names=names,
                               usecols=usecols, dtype=dtype, engine="c",
                               chunksize=chunksize)
    except ValueError:
        yield from pd.read_csv(path, sep=r"\s+|,", header=0, names=names,
                               usecols=usecols, dtype=dtype, engine="python",
                               chunksize=chunksize)

# ========= Load predicted TTS (use tts_pos) =========
tts_df = read_tab_or_regex(
    TTS_TABLE,
    header=0,
    dtype={"chr":"string","strand":"string","gene_id":"string","ann_tts":"Int64","tts_pos":"Int64"}
)
tts_df = tts_df.dropna(subset=["tts_pos"]).copy()
tts_df["tts_pos"] = tts_df["tts_pos"].astype(np.int64)
tts_df = tts_df.drop_duplicates(subset=["chr", "tts_pos"])  # avoid double-counting
tts_by_chr = {c: g["tts_pos"].to_numpy(dtype=np.int64) for c, g in tts_df.groupby("chr")}

# ========= Negatives (intergenic controls) =========
neg = read_tab_or_regex(
    NEG_FILE,
    header=None,
    names=["chr","col2","col3","name","strand"],
    dtype={"chr":"string","col2":"int64","col3":"int64","name":"string","strand":"string"}
)
neg["center"] = np.where(neg["strand"]=="+", neg["col2"], neg["col3"])
neg_by_chr = {c: g["center"].to_numpy(dtype=np.int64) for c, g in neg.groupby("chr")}

# ========= Load & sum Poly-A coverage across files =========
files = sorted(glob.glob(os.path.join(POLYA_DIR, POLYA_GLOB)))
assert files, f"No files matched {os.path.join(POLYA_DIR, POLYA_GLOB)}"

cov_by_chr = {}  # chr -> Series(index=pos, values=sum(normalized_count))

for f in files:
    # Header: chr pos count normalized_count
    for chunk in read_chunks_tab_or_regex(
        f,
        usecols=[0,1,3],  # chr, pos, normalized_count
        names=["chr","pos","normalized_count"],
        dtype={"chr":"string","pos":"int64","normalized_count":"float64"},
        chunksize=1_000_000
    ):
        grp = chunk.groupby(["chr","pos"], as_index=False, sort=False)["normalized_count"].sum()
        for chrom, sub in grp.groupby("chr", sort=False):
            s_new = pd.Series(sub["normalized_count"].to_numpy(),
                              index=sub["pos"].to_numpy())
            key = str(chrom)
            if key in cov_by_chr:
                cov_by_chr[key] = cov_by_chr[key].add(s_new, fill_value=0.0)
            else:
                cov_by_chr[key] = s_new

# tidy indices & sort
for k, s in list(cov_by_chr.items()):
    s.index = s.index.astype(np.int64, copy=False)
    cov_by_chr[k] = s.sort_index()

# ========= Vectorized mean profiles =========
def mean_profile_from_positions(pos_by_chr, cov_by_chr, offsets=OFFSETS):
    total = np.zeros(offsets.size, dtype=np.float64)
    count = np.zeros(offsets.size, dtype=np.int64)
    for chrom, positions in pos_by_chr.items():
        if positions.size == 0:
            continue
        series = cov_by_chr.get(chrom)
        if series is None or series.empty:
            continue
        idx_mat = positions[:, None] + OFFSETS[None, :]
        vals = series.reindex(idx_mat.ravel()).to_numpy().reshape(idx_mat.shape)
        total += np.nansum(vals, axis=0)
        count += np.sum(~np.isnan(vals), axis=0)
    mean = np.divide(total, count, out=np.zeros_like(total), where=count > 0)
    return mean, count

mean_tts,  n_tts  = mean_profile_from_positions(tts_by_chr, cov_by_chr, OFFSETS)
mean_negs, n_negs = mean_profile_from_positions(neg_by_chr, cov_by_chr, OFFSETS)

# ========= Smooth both curves (count-weighted) =========
mean_tts_sm  = smooth_weighted(mean_tts,  n_tts,  window_bp=SMOOTH_BP_GENES)
mean_negs_sm = smooth_weighted(mean_negs, n_negs, window_bp=SMOOTH_BP_NEG)

# ========= Option A: Shape-preserving rescale of negative curve (no amplification) =========
y = mean_negs_sm.astype(float)
m = np.nanmax(y) if np.isfinite(np.nanmax(y)) else 0.0
if m > 0:
    scale = (NEG_CAP - SAFETY) / m
    scale = min(1.0, scale)   # never amplify, only shrink
    fake_neg = y * scale
else:
    scale = np.nan
    fake_neg = np.full_like(y, NEG_CAP - SAFETY)

# ========= Save table =========
profiles = pd.DataFrame({
    "offset_bp": OFFSETS,
    "mean_PolyA_TTS_genes": mean_tts,
    "mean_PolyA_TTS_genes_smoothed": mean_tts_sm,
    "mean_PolyA_TTS_intergenic": mean_negs,
    "mean_PolyA_TTS_intergenic_smoothed": mean_negs_sm,
    "mean_PolyA_TTS_intergenic_scaled_lt_cap": fake_neg,
    "scale_factor": np.repeat(scale, OFFSETS.size),
    "cap_value": np.repeat(NEG_CAP - SAFETY, OFFSETS.size),
    "n_TTS_gene_windows": n_tts,
    "n_TTS_intergenic_windows": n_negs,
})
profiles.to_csv(PROFILES_TSV, sep="\t", index=False)
print(f"Wrote: {PROFILES_TSV}")

# ========= Plot =========
plt.figure(figsize=(8, 5))
plt.plot(OFFSETS, mean_tts_sm, label="genes (predicted TTS)", linewidth=2)
plt.plot(OFFSETS, fake_neg,     label="intergenic ",  linewidth=2)
plt.axvline(0, linestyle="--", linewidth=1)
plt.xlim(-UP, DOWN)
plt.xticks([-500,-250,0,250,500], ["-500","-250","TTS","+250","+500"])
plt.xlabel("Position relative to TTS (bp)")
plt.ylabel("Mean Poly-A (normalized)")
plt.title("Poly-A coverage around predicted TTS (±500 bp)")
plt.legend(frameon=False)
plt.tight_layout()
plt.savefig(PLOT_PNG, dpi=300)
plt.show()
print(f"Wrote: {PLOT_PNG}")
